# Run a Pathology Report from One Image
Change only **IMAGE_FILENAME** below and run the cell.

TUM image → PathGen caption → MedGemma explanation + five MUSK predictions.
The GPU job is submitted through Slurm automatically. The cell stays busy while the job queues and runs, then displays only the final report.

Use a **remote Python kernel in VS Code/Jupyter connected to Delta over SSH**. PathGen, the MedGemma base weights and adapter, and the trained MUSK checkpoint must already be available.

Default checkpoint: `../musk/runs/head_slurm_22984898/best.pt`. Explanations are in English.
Interrupting the cell requests cancellation of the submitted job. Logs and result files remain on the server.


In [ ]:
# Change only this filename, then run the cell.
IMAGE_FILENAME = "TUM-YHTSLLSL.tif"

from pathlib import Path
import sys

# Run this notebook from connect/ or the repository root.
_connect = None
for _parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    for _candidate in (_parent, _parent / "connect"):
        if (_candidate / "notebook_runner.py").is_file():
            _connect = _candidate
            break
    if _connect is not None:
        break
if _connect is None:
    raise RuntimeError("Open the connect folder on the remote server and run this notebook.")
if str(_connect) not in sys.path:
    sys.path.insert(0, str(_connect))

from notebook_runner import Settings, run_report

# The existing checkpoint and server environment are configured automatically.
# To use a different checkpoint, pass its absolute path to Settings(checkpoint=...).
settings = Settings(connect=_connect)
try:
    result = run_report(IMAGE_FILENAME, settings)
except KeyboardInterrupt:
    print("Job cancellation requested.")
except Exception as error:
    print(f"Run failed: {error}")
else:
    print(result["report"])
